In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
import zipfile
import io
import sys

sys.path.append(str(Path("..").resolve()))
from config import DATA_RAW

In [ ]:
def read_zip_csv(zip_path: Path) -> pl.LazyFrame:
    """Read the CSV inside a zip into a Polars LazyFrame."""
    with zipfile.ZipFile(zip_path) as zf:
        csv_name = [f for f in zf.namelist() if f.endswith(".csv")][0]
        with zf.open(csv_name) as f:
            raw = f.read()
    return pl.read_csv(
        io.BytesIO(raw),
        infer_schema_length=1000,
        null_values=["", "NA", "null", "NULL"],
    )


df = read_zip_csv(Path(DATA_RAW / "aisdk-2025-06-01.zip"))
df = df.filter(pl.col("Ship type").is_in(["Cargo", "Tanker", "Fishing", "Tug", "Passenger"]))

In [ ]:
df[:10]

In [ ]:
# get count of rows per vessel
vessel_counts = df.group_by(["IMO"]).agg(pl.len().alias("row_count")).sort("row_count", descending=True)

print(vessel_counts)

plt.figure(figsize=(10, 5))
plt.hist(vessel_counts["row_count"].to_numpy(), bins=10000, edgecolor="black")
plt.xlabel("Number of rows per vessel")
plt.ylabel("Number of vessels")
plt.title("Distribution of AIS pings per vessel (2024-06-04)")
plt.tight_layout()
plt.show()

In [ ]:
# get count of rows per ship type
vessel_counts_st = df.group_by(["Ship type"]).agg(pl.len().alias("row_count")).sort("row_count", descending=True)

print(vessel_counts_st)

In [ ]:
size = len(df)
print(f"""Dataset size: {size} rows \n
    Number of vessels in dataset: {len(df["IMO"].unique())}\n
    Max rows for a single vessel: {vessel_counts[0, "row_count"]}\n
    Median rows per vessel: {vessel_counts["row_count"].median()}\n
    Mean rows per vessel: {vessel_counts["row_count"].mean()}\n
    number of vessels with more than {60 * 24} rows: {(vessel_counts["row_count"] > 60 * 24).sum()}\n
    number of vessels with under {24 * 6} row: {(vessel_counts["row_count"] < 24 * 6).sum()}""")

In [ ]:
## percentage of null values per column
for col in df.columns:
    n = len(df)
    null_count = df[col].is_null().sum() / n * 100
    unknown_count = (
        ((df[col].cast(pl.Utf8) == "Unknown").sum() + (df[col].cast(pl.Utf8) == "Unknown value").sum()) / n * 100
    )
    undefined_count = (df[col].cast(pl.Utf8) == "Undefined").sum() / n * 100
    total_pct = null_count + unknown_count + undefined_count
    print(
        f"""\n{col}: \n{null_count:.2f}% null, \n{unknown_count:.2f}% unknown, \n{undefined_count:.2f}% undefined, 
        \n{total_pct:.2f}% total missing/unknown/undefined"""
    )

In [ ]:
count = df["IMO"].value_counts()
print(count)

In [ ]:
count = df["Navigational status"].value_counts().sort("count", descending=True)

for status, cnt in zip(count["Navigational status"], count["count"]):
    print(f"{status}: {cnt}")

In [ ]:
count = df["Cargo type"].value_counts().sort("count", descending=True)

for status, cnt in zip(count["Cargo type"], count["count"]):
    print(f"{status}: {cnt}")